In [4]:
!unzip /content/labels.zip


Archive:  /content/labels.zip
   creating: labels/
  inflating: labels/172427893_3_pg11.txt  
  inflating: labels/172448470_3_pg15.txt  
  inflating: labels/172561841_pg1.txt  
  inflating: labels/172566189_1_pg10.txt  
  inflating: labels/172571502_1_pg21.txt  
  inflating: labels/172576446_2_pg26.txt  
  inflating: labels/172579986_2_pg31.txt  
  inflating: labels/172585685_3_pg1.txt  
  inflating: labels/172589759_3_pg6.txt  
  inflating: labels/172607979_4_pg1.txt  
  inflating: labels/172610467_2_pg21.txt  
  inflating: labels/172611860_2_pg24.txt  
  inflating: labels/172615270_3_pg24.txt  
  inflating: labels/172615659_4_pg18.txt  
  inflating: labels/172653968_pg20.txt  
  inflating: labels/172655019_3_pg11.txt  
  inflating: labels/172656573_1_pg4.txt  
  inflating: labels/172658339_1_pg46.txt  
  inflating: labels/172674674_2_pg26.txt  
  inflating: labels/172675384_1_pg13.txt  
  inflating: labels/172676445_3_pg6.txt  
  inflating: labels/172677667_2_pg30.txt  
  inflating: 

In [3]:
!cp -r /content/drive/MyDrive/RunTimeError/Convolve/train /content

delete folders

In [ ]:
import os
import shutil

# List of paths to remove
paths_to_delete = [
    '/content/dataset',
    '/content/data.yaml'
]

for path in paths_to_delete:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)  # Deletes folder and all its contents
            print(f"✅ Deleted directory: {path}")
        else:
            os.remove(path)      # Deletes a single file
            print(f"✅ Deleted file: {path}")
    else:
        print(f"ℹ️ Not found: {path}")


✅ Deleted directory: /content/dataset
ℹ️ Not found: /content/data.yaml


In [5]:
import os
import random
import shutil
from pathlib import Path

# -------- CONFIG --------
ROOT_DIR = Path(".")              # Run from Try1/
IMAGE_SRC = ROOT_DIR / "train"
LABEL_SRC = ROOT_DIR / "labels"

SPLIT_RATIO = 0.8                 # 80% train, 20% val
CLASS_NAMES = ["sign"]

# -------- OUTPUT DIRS --------
IMG_TRAIN = ROOT_DIR / "images/train"
IMG_VAL = ROOT_DIR / "images/val"
LBL_TRAIN = ROOT_DIR / "labels/train"
LBL_VAL = ROOT_DIR / "labels/val"

for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    d.mkdir(parents=True, exist_ok=True)

# -------- COLLECT IMAGES --------
images = [
    f for f in IMAGE_SRC.iterdir()
    if f.suffix.lower() in [".png", ".jpg", ".jpeg"]
]

random.shuffle(images)

split_idx = int(len(images) * SPLIT_RATIO)
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

def move_data(img_list, img_dst, lbl_dst):
    for img_path in img_list:
        shutil.copy(img_path, img_dst / img_path.name)

        label_path = LABEL_SRC / (img_path.stem + ".txt")
        if label_path.exists():
            shutil.copy(label_path, lbl_dst / label_path.name)

# -------- SPLIT --------
move_data(train_imgs, IMG_TRAIN, LBL_TRAIN)
move_data(val_imgs, IMG_VAL, LBL_VAL)

# -------- CREATE data.yaml --------
yaml_path = ROOT_DIR / "data.yaml"

with open(yaml_path, "w") as f:
    f.write(f"""path: {ROOT_DIR.resolve()}

train: images/train
val: images/val

nc: {len(CLASS_NAMES)}
names:
""")
    for name in CLASS_NAMES:
        f.write(f"  - {name}\n")

print("✅ Dataset prepared successfully!")
print(f"📁 Train images: {len(train_imgs)}")
print(f"📁 Val images:   {len(val_imgs)}")
print("📄 data.yaml created")


✅ Dataset prepared successfully!
📁 Train images: 396
📁 Val images:   99
📄 data.yaml created


In [ ]:
!rm -rf /content/labels


In [7]:
# -------- CONFIG --------
DATASET_ROOT = "/content"
CLASS_NAMES = ["amount"]

# -------- CREATE data.yaml --------
yaml_content = f"""
path: {DATASET_ROOT}

train: images/train
val: images/val

nc: {len(CLASS_NAMES)}
names:
"""

for cls in CLASS_NAMES:
    yaml_content += f"  - {cls}\n"

# -------- SAVE FILE --------
with open("data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("✅ data.yaml created successfully!")
print("📄 Saved at: /content/data.yaml")


✅ data.yaml created successfully!
📄 Saved at: /content/data.yaml


In [8]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.8 MB/s eta 0:00:00


In [18]:
import os
import shutil

# Path to the run directory you want to delete
run_dir = '/content/runs'

# Specific weight files to delete
files_to_delete = [
    '/content/yolov8n.pt',
    '/content/yolo26n.pt'
]

# Delete the directory and all its contents
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
    print(f"Deleted directory: {run_dir}")

# Delete the specific .pt files
for file_path in files_to_delete:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleted file: {file_path}")
    else:
        print(f"File not found: {file_path}")


Deleted directory: /content/runs
Deleted file: /content/yolov8n.pt
Deleted file: /content/yolo26n.pt


In [16]:
!ls labels/train | wc -l
!ls images/train | wc -l

395
398


In [11]:
!ls labels/val | wc -l
!ls images/val | wc -l


97
97


In [15]:
import os
import shutil

val_img_dir = "images/val"
val_lbl_dir = "labels/val"
train_img_dir = "images/train"

# Get label base names
label_bases = set(
    os.path.splitext(f)[0]
    for f in os.listdir(val_lbl_dir)
    if f.endswith(".txt")
)

moved = 0

for img in os.listdir(val_img_dir):
    base = os.path.splitext(img)[0]
    if base not in label_bases:
        shutil.move(
            os.path.join(val_img_dir, img),
            os.path.join(train_img_dir, img)
        )
        moved += 1

print(f"✅ Moved {moved} extra images back to images/train")


✅ Moved 3 extra images back to images/train


In [ ]:
import torch, gc, os
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
import os
os._exit(0)


In [19]:
from ultralytics import YOLO
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=60,
    imgsz=960,
    batch=32,
    device=0,
    workers=4,
    name="amount"
)


CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=amount, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79e878ea3740>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [26]:
import shutil
import os

src = "/content/labels"
dst = "/content/drive/MyDrive/RunTimeError/Convolve/amount"

# Create destination directory if it doesn't exist
os.makedirs(dst, exist_ok=True)

# Copy entire directory tree
shutil.copytree(src, dst, dirs_exist_ok=True)


'/content/drive/MyDrive/RunTimeError/Convolve/amount'

In [23]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/amount/weights/best.pt")

results = model.predict(
    source="/content/images/val",
    conf=0.25,
    save=False
)

for r in results:
    if r.boxes is None:
        print("No detections")
        continue

    boxes = r.boxes.xyxy.cpu().numpy()
    scores = r.boxes.conf.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()

    for box, score, cls in zip(boxes, scores, classes):
        x1, y1, x2, y2 = box
        class_name = r.names[int(cls)]
        print(f"{class_name} | conf={score:.2f} | box=({int(x1)}, {int(y1)}, {int(x2)}, {int(y2)})")



image 1/97 /content/images/val/172607979_4_pg1.png: 960x704 1 amount, 88.3ms
image 2/97 /content/images/val/172615270_3_pg24.png: 960x704 1 amount, 21.8ms
image 3/97 /content/images/val/172658339_1_pg46.png: 960x768 1 amount, 155.7ms
image 4/97 /content/images/val/172684006_1_pg42.png: 960x864 1 amount, 156.8ms
image 5/97 /content/images/val/172685774_2_pg31.png: 960x704 1 amount, 12.2ms
image 6/97 /content/images/val/172689968_1_pg14.png: 960x768 1 amount, 12.0ms
image 7/97 /content/images/val/172818806_3_pg4.png: 960x704 1 amount, 11.2ms
image 8/97 /content/images/val/172820354_1_pg29.png: 960x640 1 amount, 45.4ms
image 9/97 /content/images/val/172822122_3_pg3.png: 960x704 1 amount, 11.1ms
image 10/97 /content/images/val/172903815_1_pg6.png: 960x800 2 amounts, 37.7ms
image 11/97 /content/images/val/172927572_1_pg18.png: 960x800 1 amount, 11.1ms
image 12/97 /content/images/val/172930953_3_pg2.png: 960x640 1 amount, 9.7ms
image 13/97 /content/images/val/172934848_2_pg32.png: 960x704 1

In [24]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# Load trained model
model = YOLO("/content/runs/detect/amount/weights/best.pt")

# Run prediction (no saving)
results = model.predict(
    source="/content/images/val",
    conf=0.2,
    iou=0.4,
    save=False
)

# Visualize results
for r in results:
    img = r.orig_img.copy()

    if r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        scores = r.boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):
            x1, y1, x2, y2 = map(int, box)

            # Draw bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

            # Draw label
            cv2.putText(
                img,
                f"stamp {score:.2f}",
                (x1, max(y1 - 10, 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 0),
                2
            )

    # Convert BGR → RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6, 10))
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.show()


Output hidden; open in https://colab.research.google.com to view.